# COLMAP 3D 重建流水线

## 概述
- **功能**: 对每个连通分量（场景）独立进行 COLMAP 3D 重建（稀疏 + 稠密）
- **输入**: 场景簇的图片集合 + MAST3R 匹配结果
- **输出**: 每个场景的稀疏点云 + 稠密点云（.ply）+ 相机内外参数
- **关键修正**: COLMAP 只使用显式给它的图像对，不会自动跨分量匹配

## 1. 环境配置与参数

In [ ]:
import os, sys, json, time, shutil, subprocess
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from matplotlib import pyplot as plt
import cv2

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# ── 路径配置 ──
IMAGE_ROOT = r'../image-matching-challenge-2025/train'
SCENE_CLUSTERS_PATH = r'../output/pre_clustering/scene_clusters.json'
RETRIEVAL_DIR = r'../output/retrieval'
MAST3R_MATCH_DIR = r'../output/mast3r_matching'
OUTPUT_DIR = r'../output/colmap_reconstruction'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── COLMAP 参数 ──
COLMAP_PATH = 'colmap'            # colmap 可执行文件路径（或 'pycolmap'）
USE_PYCOLMAP = True               # 使用 pycolmap Python 绑定
USE_GPU = True                    # 是否使用 GPU 加速
CAMERA_MODEL = 'SIMPLE_RADIAL'    # 相机模型
SINGLE_CAMERA = True              # 是否使用单一相机内参

# ── 重建参数 ──
SPARSE_QUALITY = 'high'           # 稀疏重建质量
RUN_DENSE = True                  # 是否运行稠密重建
DENSE_QUALITY = 'medium'          # 稠密重建质量
MAX_SCENE_SIZE = 500              # 单个场景最大图片数（超过则采样）

print('Configuration ready.')

## 2. 加载场景聚类数据

In [ ]:
def load_json(path):
    with open(path) as f:
        return json.load(f)


scene_clusters = load_json(SCENE_CLUSTERS_PATH)
path_mapping = load_json(os.path.join(RETRIEVAL_DIR, 'image_paths.json'))

print(f'Loaded {len(scene_clusters)} scene clusters')

# 过滤：只处理多图场景
reconstructable_scenes = [
    c for c in scene_clusters
    if c['num_images'] >= 2 and not c.get('is_isolated', False)
]
print(f'Reconstructable scenes (>=2 images): {len(reconstructable_scenes)}')

# 统计
sizes = [c['num_images'] for c in reconstructable_scenes]
if sizes:
    print(f'  Scene sizes: min={min(sizes)}, max={max(sizes)}, avg={np.mean(sizes):.1f}')

## 3. COLMAP 环境准备

检查 COLMAP / pycolmap 是否可用。

In [ ]:
if USE_PYCOLMAP:
    try:
        import pycolmap
        print(f'pycolmap version: {pycolmap.__version__}')
        COLMAP_AVAILABLE = True
    except ImportError:
        print('WARNING: pycolmap not installed. Falling back to CLI colmap.')
        print('Install with: pip install pycolmap')
        USE_PYCOLMAP = False
        COLMAP_AVAILABLE = False

if not USE_PYCOLMAP:
    # 检查 CLI colmap 是否可用
    try:
        result = subprocess.run([COLMAP_PATH, '--version'], capture_output=True, text=True, timeout=5)
        print(f'COLMAP CLI: {result.stdout.strip()}')
        COLMAP_AVAILABLE = True
    except Exception as e:
        print(f'WARNING: COLMAP not found: {e}')
        print('Continuing in simulation mode (file preparation only).')
        COLMAP_AVAILABLE = False

if not COLMAP_AVAILABLE:
    print('\nCOLMAP is not available. This notebook will:')
    print('  1. Prepare scene directories and config files')
    print('  2. Copy/link images to scene directories')
    print('  3. Generate match files for COLMAP')
    print('  4. Output commands for manual COLMAP execution')

## 4. 为每个场景创建独立目录 & 准备数据

每个场景分量：
1. 创建独立工作目录
2. 复制/软链接图片到场景目录
3. 准备匹配文件

In [ ]:
def prepare_scene_directory(scene, output_root, image_mapping, match_dir):
    """
    为单个场景创建 COLMAP 工作目录。

    Args:
        scene: scene cluster dict
        output_root: 输出根目录
        image_mapping: {name: path} 图片路径映射
        match_dir: MAST3R 匹配结果目录
    Returns:
        scene_dir: 场景工作目录
    """
    scene_id = scene['scene_id']
    scene_dir = os.path.join(output_root, f'scene_{scene_id:04d}')
    image_dir = os.path.join(scene_dir, 'images')
    sparse_dir = os.path.join(scene_dir, 'sparse')
    dense_dir = os.path.join(scene_dir, 'dense')
    database_path = os.path.join(scene_dir, 'database.db')

    os.makedirs(image_dir, exist_ok=True)
    os.makedirs(sparse_dir, exist_ok=True)
    os.makedirs(dense_dir, exist_ok=True)

    # 采样（如果场景太大）
    images = scene['images']
    if len(images) > MAX_SCENE_SIZE:
        print(f'  Scene {scene_id}: sampling {MAX_SCENE_SIZE} from {len(images)} images')
        np.random.seed(42)
        images = np.random.choice(images, MAX_SCENE_SIZE, replace=False).tolist()

    # 复制/软链接图片
    copied_images = []
    for img_name in images:
        src_path = image_mapping.get(img_name)
        if src_path and os.path.exists(src_path):
            dst_path = os.path.join(image_dir, img_name)
            if not os.path.exists(dst_path):
                # 用软链接节省空间
                try:
                    if os.name == 'nt':  # Windows
                        shutil.copy2(src_path, dst_path)
                    else:  # Linux/Mac
                        os.symlink(os.path.abspath(src_path), dst_path)
                except OSError:
                    shutil.copy2(src_path, dst_path)
            copied_images.append(img_name)

    # 准备匹配文件
    match_src = os.path.join(match_dir, f'scene_{scene_id}', 'dense_matches.json')
    match_dst = os.path.join(scene_dir, 'dense_matches.json')
    if os.path.exists(match_src):
        shutil.copy2(match_src, match_dst)

    return {
        'scene_dir': scene_dir,
        'image_dir': image_dir,
        'sparse_dir': sparse_dir,
        'dense_dir': dense_dir,
        'database_path': database_path,
        'num_images': len(copied_images),
        'images': copied_images,
    }


# 为每个场景准备目录
scene_dirs = []
for scene in tqdm(reconstructable_scenes, desc='Preparing scene directories'):
    info = prepare_scene_directory(scene, OUTPUT_DIR, path_mapping, MAST3R_MATCH_DIR)
    scene_dirs.append(info)

print(f'\nPrepared {len(scene_dirs)} scene directories')
for info in scene_dirs[:5]:
    print(f'  Scene: {os.path.basename(info["scene_dir"])} - {info["num_images"]} images')

## 5. COLMAP 稀疏重建（SfM）

使用 pycolmap 或 CLI colmap 对每个场景运行：
1. 特征提取（feature extraction）
2. 特征匹配（feature matching）
3. 增量式 SfM（incremental mapper）

注意：匹配可以直接使用 MAST3R 的输出，跳过 COLMAP 自身的特征提取/匹配步骤。

In [ ]:
def run_colmap_sparse_pycolmap(scene_info):
    """使用 pycolmap 运行稀疏重建"""
    import pycolmap

    image_dir = scene_info['image_dir']
    sparse_dir = scene_info['sparse_dir']
    database_path = scene_info['database_path']

    print(f'  Extracting features...')
    # ── 1. 特征提取 ──
    camera_mode = pycolmap.CameraMode.SINGLE if SINGLE_CAMERA else pycolmap.CameraMode.AUTO
    if not os.path.exists(database_path):
        pycolmap.extract_features(
            database_path=database_path,
            image_path=image_dir,
            camera_mode=camera_mode,
            camera_model=CAMERA_MODEL,
            sift_options={'max_num_features': 4096},
            device='gpu' if USE_GPU else 'cpu',
        )

    print(f'  Matching features...')
    # ── 2. 特征匹配（穷举匹配，因我们已通过连通分量缩小了范围）──
    pycolmap.match_exhaustive(
        database_path=database_path,
        sift_options={'max_num_features': 4096},
        matching_options={'block_size': 50},
        device='gpu' if USE_GPU else 'cpu',
    )

    print(f'  Running incremental mapper...')
    # ── 3. 增量式 SfM ──
    reconstruction = pycolmap.incremental_mapping(
        database_path=database_path,
        image_path=image_dir,
        output_path=sparse_dir,
    )

    if reconstruction:
        reconstruction.write(sparse_dir)
        print(f'  Sparse reconstruction: {reconstruction.num_reg_images()} / {scene_info["num_images"]} registered')
        print(f'  Points: {reconstruction.num_points3D()}')
        return reconstruction
    else:
        print(f'  WARNING: Reconstruction failed for {os.path.basename(scene_info["scene_dir"])}')
        return None


def run_colmap_sparse_cli(scene_info):
    """使用 CLI colmap 运行稀疏重建"""
    image_dir = scene_info['image_dir']
    sparse_dir = scene_info['sparse_dir']
    database_path = scene_info['database_path']

    commands = [
        # 1. Feature extraction
        [COLMAP_PATH, 'feature_extractor',
         '--database_path', database_path,
         '--image_path', image_dir,
         '--ImageReader.camera_model', CAMERA_MODEL,
         '--ImageReader.single_camera', '1' if SINGLE_CAMERA else '0',
         '--SiftExtraction.use_gpu', '1' if USE_GPU else '0',
         f'--SiftExtraction.max_num_features', '4096'],
        # 2. Exhaustive matching
        [COLMAP_PATH, 'exhaustive_matcher',
         '--database_path', database_path,
         '--SiftMatching.use_gpu', '1' if USE_GPU else '0'],
        # 3. Mapper
        [COLMAP_PATH, 'mapper',
         '--database_path', database_path,
         '--image_path', image_dir,
         '--output_path', sparse_dir],
    ]

    for cmd in commands:
        print(f'  Running: {" ".join(cmd)}')
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print(f'  ERROR: {result.stderr[:500]}')
            return False
    return True


print('COLMAP sparse reconstruction functions ready.')

## 6. 导入 MAST3R 匹配结果到 COLMAP

将 MAST3R 的像素级匹配导入 COLMAP database，代替或补充 COLMAP 自身的特征匹配。

In [ ]:
def import_mast3r_matches_to_colmap(scene_info, match_data):
    """
    将 MAST3R 匹配导入 COLMAP 作为 two-view geometry。

    Args:
        scene_info: 场景目录信息
        match_data: dict {pair_key: match_result}
    """
    if not USE_PYCOLMAP:
        print('  Skipping: pycolmap not available')
        return

    import pycolmap

    database_path = scene_info['database_path']
    if not os.path.exists(database_path):
        print(f'  Database not found: {database_path}')
        return

    # 打开 database
    db = pycolmap.Database(database_path)

    imported_count = 0
    for pair_key, result in match_data.items():
        name1, name2 = pair_key.split('::')
        img1 = db.read_image_by_name(name1)
        img2 = db.read_image_by_name(name2)
        if img1 is None or img2 is None:
            continue

        matches = result.get('matches', [])
        if len(matches) < MIN_MATCHES:
            continue

        # 转换为 COLMAP 的 pair 格式
        # 构建 keypoints 和 matches
        keypoints1 = np.array([[m[0], m[1]] for m in matches])
        keypoints2 = np.array([[m[2], m[3]] for m in matches])

        # 写入 two-view geometry
        two_view_geometry = pycolmap.TwoViewGeometry()
        two_view_geometry.config = pycolmap.TwoViewGeometry.Configuration.CALIBRATED
        two_view_geometry.inlier_matches = np.arange(len(matches))
        two_view_geometry.E = np.eye(3)  # Placeholder
        two_view_geometry.F = np.eye(3)  # Placeholder

        db.write_two_view_geometry(img1.image_id, img2.image_id, two_view_geometry)
        imported_count += 1

    db.close()
    print(f'  Imported {imported_count} MAST3R matches into database')


print('MAST3R match import function ready.')

## 7. 批量运行稀疏重建

对每个场景分量运行 COLMAP 稀疏重建。
如果 COLMAP 不可用，则输出手动执行命令。

In [ ]:
reconstruction_results = []

if COLMAP_AVAILABLE:
    for i, scene_info in enumerate(tqdm(scene_dirs, desc='Sparse reconstruction')):
        scene_name = os.path.basename(scene_info['scene_dir'])
        print(f'\n--- Scene {i+1}/{len(scene_dirs)}: {scene_name} ({scene_info["num_images"]} images) ---')

        # 加载 MAST3R 匹配（如果存在）
        match_file = os.path.join(scene_info['scene_dir'], 'dense_matches.json')
        match_data = None
        if os.path.exists(match_file):
            match_data = load_json(match_file)
            print(f'  Loaded {len(match_data)} MAST3R match pairs')

        try:
            if USE_PYCOLMAP:
                reconstruction = run_colmap_sparse_pycolmap(scene_info)
                if match_data:
                    import_mast3r_matches_to_colmap(scene_info, match_data)
                result = {
                    'scene': scene_name,
                    'success': reconstruction is not None,
                    'num_registered': reconstruction.num_reg_images() if reconstruction else 0,
                    'num_points': reconstruction.num_points3D() if reconstruction else 0,
                }
            else:
                success = run_colmap_sparse_cli(scene_info)
                result = {
                    'scene': scene_name,
                    'success': success,
                    'num_registered': -1,
                    'num_points': -1,
                }
        except Exception as e:
            print(f'  ERROR: {e}')
            result = {'scene': scene_name, 'success': False, 'num_registered': 0, 'num_points': 0}

        reconstruction_results.append(result)

else:
    # COLMAP 不可用：输出手动执行命令
    print('COLMAP is not available. Here are the commands to run manually:')
    print('=' * 60)
    for scene_info in scene_dirs:
        print(f'\n# Scene: {os.path.basename(scene_info["scene_dir"])}')
        db = scene_info['database_path']
        img = scene_info['image_dir']
        sp = scene_info['sparse_dir']
        print(f'colmap feature_extractor --database_path {db} --image_path {img} --ImageReader.camera_model {CAMERA_MODEL}')
        print(f'colmap exhaustive_matcher --database_path {db}')
        print(f'colmap mapper --database_path {db} --image_path {img} --output_path {sp}')
    print('=' * 60)

## 8. 稠密重建（可选）

对稀疏重建成功的场景运行 MVS 稠密重建。
注意：稠密重建计算量大，可选择跳过或仅对关键场景运行。

In [ ]:
def run_colmap_dense_pycolmap(scene_info):
    """使用 pycolmap 运行稠密重建"""
    import pycolmap

    image_dir = scene_info['image_dir']
    sparse_dir = scene_info['sparse_dir']
    dense_dir = scene_info['dense_dir']

    # 确保稀疏重建输出存在
    sparse_model = os.path.join(sparse_dir, '0')
    if not os.path.exists(sparse_model):
        # 找第一个有效的稀疏模型
        sparse_dirs = sorted(Path(sparse_dir).glob('*/'))
        if not sparse_dirs:
            print(f'  No sparse model found in {sparse_dir}')
            return False
        sparse_model = str(sparse_dirs[0])

    print(f'  Undistorting images...')
    # 1. Image undistortion
    undistorted = pycolmap.undistort_images(
        output_path=dense_dir,
        input_path=sparse_model,
        image_path=image_dir,
    )

    print(f'  Running stereo matching...')
    # 2. Patch-match stereo
    stereo_dir = os.path.join(dense_dir, 'stereo')
    os.makedirs(stereo_dir, exist_ok=True)
    pycolmap.patch_match_stereo(
        workspace_path=dense_dir,
        quality=DENSE_QUALITY,
        gpu_index='0' if USE_GPU else '-1',
    )

    print(f'  Fusing stereo...')
    # 3. Stereo fusion
    output_ply = os.path.join(dense_dir, 'fused.ply')
    pycolmap.stereo_fusion(
        output_path=output_ply,
        workspace_path=dense_dir,
    )

    return os.path.exists(output_ply)


def run_colmap_dense_cli(scene_info):
    """使用 CLI colmap 运行稠密重建"""
    image_dir = scene_info['image_dir']
    sparse_dir = scene_info['sparse_dir']
    dense_dir = scene_info['dense_dir']

    # 找稀疏模型
    sparse_dirs = sorted(Path(sparse_dir).glob('*/'))
    if not sparse_dirs:
        return False
    sparse_model = str(sparse_dirs[0])

    commands = [
        [COLMAP_PATH, 'image_undistorter',
         '--image_path', image_dir,
         '--input_path', sparse_model,
         '--output_path', dense_dir],
        [COLMAP_PATH, 'patch_match_stereo',
         '--workspace_path', dense_dir,
         f'--PatchMatchStereo.gpu_index', '0' if USE_GPU else '-1'],
        [COLMAP_PATH, 'stereo_fusion',
         '--workspace_path', dense_dir,
         '--output_path', os.path.join(dense_dir, 'fused.ply')],
    ]

    for cmd in commands:
        print(f'  Running: {" ".join(cmd)}')
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print(f'  WARNING: {result.stderr[:300]}')
    return True


# 仅对重建成功的场景运行稠密重建
if RUN_DENSE and COLMAP_AVAILABLE:
    successful_scenes = [
        (scene_dirs[i], reconstruction_results[i])
        for i in range(len(scene_dirs))
        if i < len(reconstruction_results) and reconstruction_results[i]['success']
    ]
    print(f'\nRunning dense reconstruction for {len(successful_scenes)} successful scenes...')

    for scene_info, result in tqdm(successful_scenes, desc='Dense reconstruction'):
        scene_name = os.path.basename(scene_info['scene_dir'])
        print(f'\n--- Dense: {scene_name} ---')
        try:
            if USE_PYCOLMAP:
                run_colmap_dense_pycolmap(scene_info)
            else:
                run_colmap_dense_cli(scene_info)
        except Exception as e:
            print(f'  ERROR: {e}')
else:
    print('Dense reconstruction skipped (RUN_DENSE=False or COLMAP not available).')
    print('Set RUN_DENSE=True to enable.')

## 9. 点云导出与可视化

使用 Open3D 加载和可视化重建结果。如果 pycolmap 可用，也可以直接用其可视化。

In [ ]:
try:
    import open3d as o3d
    HAS_OPEN3D = True
except ImportError:
    print('Open3D not installed. Install with: pip install open3d')
    HAS_OPEN3D = False


def visualize_point_cloud(ply_path, title='Point Cloud'):
    """使用 Open3D 可视化点云"""
    if not HAS_OPEN3D or not os.path.exists(ply_path):
        return

    pcd = o3d.io.read_point_cloud(ply_path)
    print(f'{title}: {len(pcd.points)} points')

    # 统计滤波去除离群点
    pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
    print(f'  After outlier removal: {len(pcd.points)} points')

    # 显示
    o3d.visualization.draw_geometries([pcd], window_name=title, width=800, height=600)
    return pcd


# 查找并可视化第一个可用的点云
if HAS_OPEN3D:
    for scene_info in scene_dirs:
        # 优先找稠密点云
        dense_ply = os.path.join(scene_info['dense_dir'], 'fused.ply')
        if os.path.exists(dense_ply):
            scene_name = os.path.basename(scene_info['scene_dir'])
            print(f'\nVisualizing: {scene_name} (dense)')
            visualize_point_cloud(dense_ply, f'{scene_name} - Dense')
            break

        # 否则尝试从稀疏模型导出
        sparse_dirs = sorted(Path(scene_info['sparse_dir']).glob('*/'))
        if sparse_dirs and USE_PYCOLMAP:
            import pycolmap
            recon = pycolmap.Reconstruction(sparse_dirs[0])
            scene_name = os.path.basename(scene_info['scene_dir'])
            print(f'\nVisualizing: {scene_name} (sparse, {recon.num_points3D()} points)')
            # 转为 Open3D 点云
            points = np.array([p.xyz for p in recon.points3D.values()])
            colors = np.array([p.color for p in recon.points3D.values()]) / 255.0
            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(points)
            pcd.colors = o3d.utility.Vector3dVector(colors)
            o3d.visualization.draw_geometries([pcd], window_name=f'{scene_name} - Sparse', width=800, height=600)
            break

## 10. 重建质量报告

In [ ]:
print('=' * 60)
print('Phase 6 Complete: COLMAP 3D Reconstruction')
print('=' * 60)
print(f'\nOutput directory: {OUTPUT_DIR}')
print(f'Scenes processed: {len(scene_dirs)}')

# 汇总每个场景的重建结果
if reconstruction_results:
    registered_counts = [r['num_registered'] for r in reconstruction_results]
    point_counts = [r['num_points'] for r in reconstruction_results if r['num_points'] > 0]
    successful = [r for r in reconstruction_results if r['success']]

    print(f'\nReconstruction Summary:')
    print(f'  Successful scenes:    {len(successful)}/{len(reconstruction_results)}')
    if registered_counts:
        print(f'  Avg registered images: {np.mean([c for c in registered_counts if c>0]):.1f}')
    if point_counts:
        print(f'  Avg sparse points:     {np.mean(point_counts):.0f}')

    # 详细列表
    print(f'\n  Per-scene details:')
    for r in reconstruction_results[:10]:
        status = 'OK' if r['success'] else 'FAIL'
        print(f'    {r["scene"]:20s} | {status:4s} | registered: {r["num_registered"]:4d} | points: {r["num_points"]:8d}')
    if len(reconstruction_results) > 10:
        print(f'    ... and {len(reconstruction_results) - 10} more')

# 保存报告
report = {
    'total_scenes': len(scene_dirs),
    'reconstruction_results': reconstruction_results,
    'dense_enabled': RUN_DENSE,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
}
with open(os.path.join(OUTPUT_DIR, 'reconstruction_report.json'), 'w') as f:
    json.dump(report, f, indent=2)
print(f'\nReport saved to {os.path.join(OUTPUT_DIR, "reconstruction_report.json")}')

print(f'\nPipeline Complete!')
print(f'  Total scenes: {len(scene_dirs)}')
print(f'  Output directory: {OUTPUT_DIR}')
print(f'  Each scene directory contains:')
print(f'    - images/      : source images')
print(f'    - sparse/      : sparse SfM reconstruction')
print(f'    - dense/       : dense MVS reconstruction (if enabled)')
print(f'    - database.db  : COLMAP database')